# USDA SSURGO Soil Data Access for CropSage

This notebook is the CropSage prototype for the USDA NRCS **Soil Data Access (SDA)**
query service. It looks up the SSURGO map unit at a farm coordinate, retrieves major
soil components and horizons, creates a transparent soil-evidence summary, and compares
only the relevant soil fields with the 22-crop catalog.

**Endpoint used:** `POST https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest`

The endpoint accepts a SQL query in a JSON body and does not require an API key. CropSage
requests `JSON+COLUMNNAME`, for which the first returned row contains column names.

Official references:

- [Soil Data Access Web Service Help](https://sdmdataaccess.nrcs.usda.gov/WebServiceHelp.aspx)
- [SDA spatial query functions and WKT examples](https://sdmdataaccess.nrcs.usda.gov/documents/AdvancedQueries.html)
- [Current Soil Data Mart tables and columns](https://sdmdataaccess.nrcs.usda.gov/documents/TablesAndColumnsReport.pdf)
- [NRCS SSURGO overview](https://www.nrcs.usda.gov/resources/data-and-reports/soil-survey-geographic-database-ssurgo)

> SSURGO is mapped survey evidence, not a live sensor or a laboratory test. CropSage
> should use it as default soil evidence and let a farmer observation or laboratory result
> override the corresponding mapped value while retaining both sources.

## Fields CropSage will use from this endpoint

| Decision area | Crop-catalog field | Farm-profile field | SSURGO columns | CropSage use |
|---|---|---|---|---|
| Location lookup | Supported Texas region is handled elsewhere | `latitude`, `longitude` | Point WKT passed to `SDA_Get_Mukey_from_intersection_with_WktWgs84` | Find the mapped soil unit at the selected farm point |
| Soil texture | `preferred_soil_textures` | `known soil texture` (optional override) | `chtexturegrp.texture`, `sandtotal_r`, `silttotal_r`, `claytotal_r` | Soft texture-fit comparison; retain conflicts and prefer field/lab evidence |
| Soil pH | `ph_tolerable_range` | `laboratory pH` (optional override) | `chorizon.ph1to1h2o_r` | Compare a 0-30 cm depth-weighted mapped estimate with each crop's tolerated range; lab pH overrides |
| Drainage | `drainage_requirement` | No current direct input in the finalized profile table | `component.drainagecl`, with `hydricrating` as context | Add mapped drainage evidence and flag likely mismatches without treating it as field observation |
| Soil water storage | `effective_root_zone_depth_cm`; `water_demand` and `drought_tolerance` remain context | Irrigation inputs remain farmer supplied | `chorizon.awc_r`, horizon depths, and `corestrictions.resdept_r` | Estimate accessible storage across each crop's min-max root zone after applying the mapped soil-depth limit |
| Provenance | Evidence/quality metadata | Input source labels | `areasymbol`, `mukey`, `musym`, `muname`, `saverest`, component percentage | Make the result auditable and support confidence notes |

### Fields this endpoint does **not** supply

- Crop temperature limits, planting windows, days to maturity, frost sensitivity, or heat-stage rules.
- Seasonal rainfall, recent rainfall, current soil moisture, or weather forecasts.
- Irrigation availability/reliability, well pumping capacity, or canal allocation.
- A laboratory pH measurement or exact within-field soil condition.
- The farmer's planting date, goal, or management decisions.

The catalog's `water_demand.seasonal_range_mm` must therefore be compared with climate,
recent-weather, and irrigation evidence. SSURGO available-water capacity only describes
how much plant-available water the mapped soil can store across the crop-accessible depth.
The usable depth is the smaller of the catalog crop depth and the mapped soil-depth limit.

## 1. Setup and sample farm point

The default point is a rural High Plains location near Lubbock, Texas. Replace it with the
farm's latitude and longitude. WKT uses **longitude first**, then latitude.

In [1]:
from pathlib import Path
import json
import math

import pandas as pd
import requests
from IPython.display import display

SDA_ENDPOINT = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"

# Rural Texas High Plains sample point.
LATITUDE = 33.4000
LONGITUDE = -101.8000

TOPSOIL_DEPTH_CM = 30
SSURGO_FALLBACK_ROOT_LIMIT_CM = 150
REQUEST_TIMEOUT_SECONDS = 45

## 2. Build and call the SDA query

The query joins the point's map unit to its major components, horizons, and representative
texture group. CropSage keeps every returned horizon so its derived values can be audited.
Coordinates are validated and formatted as numbers before insertion into SQL.

In [2]:
def validate_coordinate(latitude: float, longitude: float) -> tuple[float, float]:
    latitude = float(latitude)
    longitude = float(longitude)
    if not (math.isfinite(latitude) and math.isfinite(longitude)):
        raise ValueError("Latitude and longitude must be finite numbers.")
    if not -90 <= latitude <= 90:
        raise ValueError("Latitude must be between -90 and 90.")
    if not -180 <= longitude <= 180:
        raise ValueError("Longitude must be between -180 and 180.")
    return latitude, longitude


def build_point_soil_query(latitude: float, longitude: float) -> str:
    latitude, longitude = validate_coordinate(latitude, longitude)
    point_wkt = f"POINT({longitude:.8f} {latitude:.8f})"
    return f"""
SELECT
  l.areasymbol,
  sac.saverest,
  mu.mukey,
  mu.musym,
  mu.muname,
  mu.farmlndcl,
  co.cokey,
  co.compname,
  co.comppct_r,
  co.majcompflag,
  co.drainagecl,
  co.hydricrating,
  cr.restrictive_depth_cm,
  ch.chkey,
  ch.hzname,
  ch.hzdept_r,
  ch.hzdepb_r,
  ch.awc_r,
  ch.ph1to1h2o_r,
  ch.sandtotal_r,
  ch.silttotal_r,
  ch.claytotal_r,
  tx.texture
FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('{point_wkt}') AS p
JOIN mapunit AS mu ON mu.mukey = p.mukey
JOIN legend AS l ON l.lkey = mu.lkey
LEFT JOIN sacatalog AS sac ON sac.areasymbol = l.areasymbol
JOIN component AS co ON co.mukey = mu.mukey
LEFT JOIN (
  SELECT cokey, MIN(resdept_r) AS restrictive_depth_cm
  FROM corestrictions
  WHERE resdept_r IS NOT NULL
  GROUP BY cokey
) AS cr ON cr.cokey = co.cokey
JOIN chorizon AS ch ON ch.cokey = co.cokey
LEFT JOIN chtexturegrp AS tx
  ON tx.chkey = ch.chkey AND tx.rvindicator = 'Yes'
WHERE co.majcompflag = 'Yes'
ORDER BY co.comppct_r DESC, co.cokey, ch.hzdept_r
""".strip()


def parse_sda_table(payload: dict) -> pd.DataFrame:
    table = payload.get("Table")
    if not table:
        raise ValueError(f"SDA returned no Table data: {payload}")
    columns, *rows = table
    return pd.DataFrame(rows, columns=columns)


def fetch_point_soils(
    latitude: float,
    longitude: float,
    timeout: int = REQUEST_TIMEOUT_SECONDS,
) -> tuple[pd.DataFrame, str]:
    query = build_point_soil_query(latitude, longitude)
    response = requests.post(
        SDA_ENDPOINT,
        json={"query": query, "format": "JSON+COLUMNNAME"},
        timeout=timeout,
    )
    response.raise_for_status()
    frame = parse_sda_table(response.json())
    if frame.empty:
        raise LookupError(
            "No SSURGO major-component horizons were returned for this point. "
            "Check that the coordinate is in covered U.S. soil-survey data."
        )
    return frame, query

In [3]:
raw_soils, submitted_query = fetch_point_soils(LATITUDE, LONGITUDE)

numeric_columns = [
    "comppct_r", "restrictive_depth_cm", "hzdept_r", "hzdepb_r", "awc_r", "ph1to1h2o_r",
    "sandtotal_r", "silttotal_r", "claytotal_r",
]
for column in numeric_columns:
    raw_soils[column] = pd.to_numeric(raw_soils[column], errors="coerce")

print(f"Returned {len(raw_soils)} major-component horizon rows.")
display(raw_soils)

Returned 5 major-component horizon rows.


,areasymbol,saverest,mukey,musym,muname,farmlndcl,cokey,compname,comppct_r,majcompflag,...,chkey,hzname,hzdept_r,hzdepb_r,awc_r,ph1to1h2o_r,sandtotal_r,silttotal_r,claytotal_r,texture
0,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954694,Ap,0,15,0.17,7.9,34.7,37.2,28.1,CL
1,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954690,Bt1,15,48,0.15,8.0,33.6,32.5,33.9,CL
2,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954691,Bt2,48,97,0.15,8.0,33.6,32.5,33.9,CL
3,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954692,Btk,97,127,0.14,8.2,38.3,24.0,37.7,CL
4,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954693,Btkk,127,203,0.15,8.3,40.2,20.3,39.5,CL


## 3. Normalize point evidence

SSURGO map units can contain several components. For a point lookup, this prototype uses
the major component with the largest representative percentage and reports that percentage.
It does not imply that the exact point was field-sampled.

- **Surface texture:** representative texture of the shallowest returned horizon.
- **Mapped pH:** horizon-thickness-weighted mean from 0-30 cm.
- **Soil root-depth limit:** shallowest `corestrictions.resdept_r` value for the selected
  component. When SSURGO reports no restriction, CropSage uses the documented 150 cm
  gSSURGO approximation and labels it as a fallback.
- **Available water:** integrated from horizon `awc_r` values to the requested crop depth,
  after capping that depth at the soil root-depth limit. Horizon coverage is always reported.

In [4]:
TEXTURE_CODE_TO_NAME = {
    "S": "sand",
    "LS": "loamy sand",
    "SL": "sandy loam",
    "L": "loam",
    "SIL": "silt loam",
    "SI": "silt",
    "SCL": "sandy clay loam",
    "CL": "clay loam",
    "SICL": "silty clay loam",
    "SC": "sandy clay",
    "SIC": "silty clay",
    "C": "clay",
}


def overlap_cm(top: float, bottom: float, start: float, end: float) -> float:
    return max(0.0, min(float(bottom), end) - max(float(top), start))


def weighted_horizon_mean(
    horizons: pd.DataFrame,
    value_column: str,
    depth_cm: float,
) -> tuple[float | None, float]:
    weighted_sum = 0.0
    covered_cm = 0.0
    for _, row in horizons.iterrows():
        if pd.isna(row[value_column]) or pd.isna(row["hzdept_r"]) or pd.isna(row["hzdepb_r"]):
            continue
        thickness = overlap_cm(row["hzdept_r"], row["hzdepb_r"], 0, depth_cm)
        weighted_sum += float(row[value_column]) * thickness
        covered_cm += thickness
    if covered_cm == 0:
        return None, 0.0
    return weighted_sum / covered_cm, covered_cm


def available_water_mm(horizons: pd.DataFrame, depth_cm: float) -> tuple[float | None, float]:
    water_mm = 0.0
    covered_cm = 0.0
    for _, row in horizons.iterrows():
        if pd.isna(row["awc_r"]) or pd.isna(row["hzdept_r"]) or pd.isna(row["hzdepb_r"]):
            continue
        thickness = overlap_cm(row["hzdept_r"], row["hzdepb_r"], 0, depth_cm)
        water_mm += float(row["awc_r"]) * thickness * 10.0
        covered_cm += thickness
    if covered_cm == 0:
        return None, 0.0
    return water_mm, covered_cm


def summarize_dominant_component(
    soils: pd.DataFrame,
    topsoil_depth_cm: float = TOPSOIL_DEPTH_CM,
    fallback_root_limit_cm: float = SSURGO_FALLBACK_ROOT_LIMIT_CM,
) -> tuple[dict, pd.DataFrame]:
    component_order = (
        soils[["cokey", "comppct_r"]]
        .drop_duplicates()
        .sort_values("comppct_r", ascending=False, na_position="last")
    )
    dominant_cokey = component_order.iloc[0]["cokey"]
    horizons = (
        soils[soils["cokey"] == dominant_cokey]
        .sort_values("hzdept_r")
        .drop_duplicates(subset=["chkey"])
        .copy()
    )
    first = horizons.iloc[0]
    texture_code = str(first["texture"]).strip().upper() if pd.notna(first["texture"]) else None
    texture_name = TEXTURE_CODE_TO_NAME.get(texture_code, texture_code.lower() if texture_code else None)
    ph_mean, ph_coverage = weighted_horizon_mean(horizons, "ph1to1h2o_r", topsoil_depth_cm)
    restriction_values = horizons["restrictive_depth_cm"].dropna()
    if restriction_values.empty:
        soil_root_limit_cm = float(fallback_root_limit_cm)
        soil_root_limit_basis = "SSURGO reported no restriction; gSSURGO 150 cm fallback"
    else:
        soil_root_limit_cm = float(restriction_values.min())
        soil_root_limit_basis = "shallowest SSURGO component restriction"
    awc_mm, awc_coverage = available_water_mm(horizons, soil_root_limit_cm)

    summary = {
        "source": "USDA NRCS SSURGO via Soil Data Access",
        "query_latitude": LATITUDE,
        "query_longitude": LONGITUDE,
        "survey_area_symbol": first["areasymbol"],
        "survey_data_saved": first["saverest"],
        "map_unit_key": first["mukey"],
        "map_unit_symbol": first["musym"],
        "map_unit_name": first["muname"],
        "farmland_classification": first["farmlndcl"],
        "dominant_component": first["compname"],
        "dominant_component_percent": first["comppct_r"],
        "surface_texture_code": texture_code,
        "surface_texture": texture_name,
        "surface_sand_percent": first["sandtotal_r"],
        "surface_silt_percent": first["silttotal_r"],
        "surface_clay_percent": first["claytotal_r"],
        f"mapped_ph_0_{int(topsoil_depth_cm)}cm": round(ph_mean, 2) if ph_mean is not None else None,
        "ph_depth_coverage_cm": round(ph_coverage, 1),
        "drainage_class": first["drainagecl"],
        "hydric_rating": first["hydricrating"],
        "soil_root_limit_cm": round(soil_root_limit_cm, 1),
        "soil_root_limit_basis": soil_root_limit_basis,
        "available_water_mm_to_soil_limit": round(awc_mm, 1) if awc_mm is not None else None,
        "available_water_depth_coverage_cm": round(awc_coverage, 1),
    }
    return summary, horizons


soil_summary, dominant_horizons = summarize_dominant_component(raw_soils)
display(pd.Series(soil_summary, name="value").to_frame())
display(dominant_horizons)

,value
source,USDA NRCS SSURGO via Soil Data Access
query_latitude,33.4
query_longitude,-101.8
survey_area_symbol,TX303
survey_data_saved,9/4/2025 7:34:30 PM
map_unit_key,369837
map_unit_symbol,EcA
map_unit_name,"Estacado clay loam, 0 to 1 percent slopes"
farmland_classification,All areas are prime farmland
dominant_component,Estacado


,areasymbol,saverest,mukey,musym,muname,farmlndcl,cokey,compname,comppct_r,majcompflag,...,chkey,hzname,hzdept_r,hzdepb_r,awc_r,ph1to1h2o_r,sandtotal_r,silttotal_r,claytotal_r,texture
0,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954694,Ap,0,15,0.17,7.9,34.7,37.2,28.1,CL
1,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954690,Bt1,15,48,0.15,8.0,33.6,32.5,33.9,CL
2,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954691,Bt2,48,97,0.15,8.0,33.6,32.5,33.9,CL
3,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954692,Btk,97,127,0.14,8.2,38.3,24.0,37.7,CL
4,TX303,9/4/2025 7:34:30 PM,369837,EcA,"Estacado clay loam, 0 to 1 percent slopes",All areas are prime farmland,27108846,Estacado,85,Yes,...,80954693,Btkk,127,203,0.15,8.3,40.2,20.3,39.5,CL


## 4. Compare SSURGO evidence with the 22-crop catalog

This is a transparent field-level comparison, not the final CropSage suitability score.
Texture, pH, and drainage are soft evidence. For water storage, each crop contributes a
sourced minimum and maximum effective rooting depth. CropSage caps both at the SSURGO
soil-depth limit and integrates horizon available-water capacity to each resulting depth.

The output is an **accessible soil-water storage range**, not a seasonal water-supply result.
Rainfall, current moisture, irrigation availability, and crop seasonal demand remain separate.

In [5]:
def find_repository_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "crop-catalog" / "catalog.json").exists():
            return candidate
    raise FileNotFoundError("Could not find data/crop-catalog/catalog.json")


REPOSITORY_ROOT = find_repository_root()
CATALOG_PATH = REPOSITORY_ROOT / "data" / "crop-catalog" / "catalog.json"
with CATALOG_PATH.open("r", encoding="utf-8") as handle:
    catalog = json.load(handle)

print(f"Catalog version {catalog['catalog_version']}: {len(catalog['crops'])} crops")

Catalog version 1.1.0: 22 crops


In [6]:
def canonical_texture(value: str | None) -> str | None:
    if not value:
        return None
    text = str(value).lower().replace("-", " ").strip()
    for qualifier in ("well drained ", "light textured ", "deep "):
        if text.startswith(qualifier):
            text = text[len(qualifier):]
    return " ".join(text.split())


def ph_fit(value: float | None, requirement: dict) -> str:
    if value is None:
        return "unknown"
    if value < requirement["min"]:
        return "below range"
    if value > requirement["max"]:
        return "above range"
    return "within range"


def drainage_fit(actual: str | None, requirement_class: str) -> str:
    if not actual:
        return "unknown"
    rank = {
        "very poorly drained": 0,
        "poorly drained": 1,
        "somewhat poorly drained": 2,
        "moderately well drained": 3,
        "well drained": 4,
        "somewhat excessively drained": 5,
        "excessively drained": 6,
    }.get(actual.lower())
    if rank is None:
        return "review"
    if requirement_class == "poorly_drained_tolerant":
        return "compatible"
    if requirement_class == "moderate":
        return "compatible" if 2 <= rank <= 4 else "review"
    if requirement_class == "well_drained":
        return "compatible" if rank >= 3 else "likely mismatch"
    if requirement_class == "very_well_drained":
        return "compatible" if rank >= 4 else "likely mismatch"
    return "review"


def format_range(minimum: float | None, maximum: float | None, decimals: int = 1) -> str | None:
    if minimum is None or maximum is None:
        return None
    return f"{minimum:.{decimals}f}-{maximum:.{decimals}f}"


def compare_catalog_to_soil(
    crops: list[dict],
    summary: dict,
    horizons: pd.DataFrame,
) -> pd.DataFrame:
    soil_texture = canonical_texture(summary["surface_texture"])
    mapped_ph = summary[f"mapped_ph_0_{TOPSOIL_DEPTH_CM}cm"]
    soil_limit_cm = float(summary["soil_root_limit_cm"])
    rows = []
    for crop in crops:
        preferred = {canonical_texture(item) for item in crop["preferred_soil_textures"]}
        crop_depth = crop["effective_root_zone_depth_cm"]
        usable_min_cm = min(float(crop_depth["min"]), soil_limit_cm)
        usable_max_cm = min(float(crop_depth["max"]), soil_limit_cm)
        storage_min_mm, coverage_min_cm = available_water_mm(horizons, usable_min_cm)
        storage_max_mm, coverage_max_cm = available_water_mm(horizons, usable_max_cm)
        rows.append(
            {
                "crop_id": crop["crop_id"],
                "crop": crop["common_name"],
                "SSURGO_texture": soil_texture,
                "texture_fit": "preferred" if soil_texture in preferred else "not listed as preferred",
                "mapped_pH": mapped_ph,
                "catalog_pH_range": (
                    f"{crop['ph_tolerable_range']['min']}-"
                    f"{crop['ph_tolerable_range']['max']}"
                ),
                "pH_fit": ph_fit(mapped_ph, crop["ph_tolerable_range"]),
                "SSURGO_drainage": summary["drainage_class"],
                "drainage_fit": drainage_fit(
                    summary["drainage_class"],
                    crop["drainage_requirement"]["class"],
                ),
                "catalog_root_zone_cm": f"{crop_depth['min']}-{crop_depth['max']}",
                "soil_root_limit_cm": soil_limit_cm,
                "usable_root_zone_cm": format_range(usable_min_cm, usable_max_cm, 0),
                "accessible_water_storage_mm": format_range(storage_min_mm, storage_max_mm),
                "water_storage_coverage_cm": format_range(coverage_min_cm, coverage_max_cm, 0),
                "water_use_rule": "storage context only; not rainfall or irrigation supply",
            }
        )
    return pd.DataFrame(rows)


catalog_comparison = compare_catalog_to_soil(
    catalog["crops"],
    soil_summary,
    dominant_horizons,
)
display(catalog_comparison)

,crop_id,crop,SSURGO_texture,texture_fit,mapped_pH,catalog_pH_range,pH_fit,SSURGO_drainage,drainage_fit,catalog_root_zone_cm,soil_root_limit_cm,usable_root_zone_cm,accessible_water_storage_mm,water_storage_coverage_cm,water_use_rule
0,upland_cotton,Upland cotton,clay loam,preferred,7.95,5.7-7.0,above range,Well drained,compatible,100-170,150.0,100-150,152.7-225.0,100-150,storage context only; not rainfall or irrigati...
1,corn_grain,Corn grown for grain,clay loam,preferred,7.95,5.5-7.0,above range,Well drained,compatible,100-170,150.0,100-150,152.7-225.0,100-150,storage context only; not rainfall or irrigati...
2,hard_red_winter_wheat,Hard red winter wheat,clay loam,preferred,7.95,5.5-7.0,above range,Well drained,compatible,150-180,150.0,150-150,225.0-225.0,150-150,storage context only; not rainfall or irrigati...
3,grain_sorghum,Grain sorghum,clay loam,preferred,7.95,5.5-7.0,above range,Well drained,compatible,100-200,150.0,100-150,152.7-225.0,100-150,storage context only; not rainfall or irrigati...
4,runner_peanut,Runner-type peanut,clay loam,not listed as preferred,7.95,5.8-7.0,above range,Well drained,compatible,50-100,150.0,50-100,78.0-152.7,50-100,storage context only; not rainfall or irrigati...
5,long_grain_rice,Long-grain rice,clay loam,preferred,7.95,5.5-7.0,above range,Well drained,compatible,50-100,150.0,50-100,78.0-152.7,50-100,storage context only; not rainfall or irrigati...
6,soybean,Soybean,clay loam,preferred,7.95,5.8-7.0,above range,Well drained,compatible,60-130,150.0,60-130,93.0-195.0,60-130,storage context only; not rainfall or irrigati...
7,grain_oats,Grain oats,clay loam,preferred,7.95,5.5-7.0,above range,Well drained,compatible,100-150,150.0,100-150,152.7-225.0,100-150,storage context only; not rainfall or irrigati...
8,oilseed_sunflower,Oilseed sunflower,clay loam,preferred,7.95,5.5-7.5,above range,Well drained,compatible,80-150,150.0,80-150,123.0-225.0,80-150,storage context only; not rainfall or irrigati...
9,sesame,Sesame,clay loam,not listed as preferred,7.95,5.5-8.0,within range,Well drained,compatible,100-150,150.0,100-150,152.7-225.0,100-150,storage context only; not rainfall or irrigati...


In [7]:
# Contract checks for the notebook pipeline. These verify shape and units, not agronomic truth.
assert catalog["catalog_version"] == "1.1.0"
assert len(catalog_comparison) == len(catalog["crops"]) == 22
assert raw_soils["mukey"].notna().all()
assert dominant_horizons["hzdept_r"].is_monotonic_increasing
assert 0 <= float(soil_summary["dominant_component_percent"]) <= 100
assert soil_summary["ph_depth_coverage_cm"] <= TOPSOIL_DEPTH_CM
assert soil_summary["available_water_depth_coverage_cm"] <= soil_summary["soil_root_limit_cm"]
assert all(
    crop["effective_root_zone_depth_cm"]["min"]
    <= crop["effective_root_zone_depth_cm"]["max"]
    for crop in catalog["crops"]
)
assert set(catalog_comparison["pH_fit"]) <= {
    "below range", "within range", "above range", "unknown"
}
print("Notebook contract checks passed.")

Notebook contract checks passed.


## 5. Runtime rules for CropSage

1. Use latitude/longitude to retrieve the mapped point evidence. A future field-boundary
   workflow can use the same SDA service with polygon WKT, but it must area-weight map units
   and components rather than treating every intersected map unit equally.
2. Preserve raw map-unit, component, restriction, and horizon rows with the query coordinates,
   retrieval time, and SSURGO `saverest` value.
3. Treat the dominant-component percentage as an uncertainty signal. A mapped component is
   not proof of exact field conditions.
4. Compare surface texture, 0-30 cm mapped pH, and drainage with the catalog as soft evidence.
5. For each crop, cap both catalog root-depth bounds at the shallowest SSURGO component
   restriction. If no restriction is reported, use and label the gSSURGO 150 cm fallback.
6. Integrate `awc_r` separately to the usable minimum and maximum depths. Report incomplete
   horizon coverage instead of extrapolating missing water capacity.
7. If the farmer provides known texture, retain both values and prefer the farmer/field value
   while flagging disagreement. If laboratory pH or a known restrictive depth is supplied, it
   overrides the corresponding mapped estimate while both sources remain visible.
8. Accessible soil-water storage does not provide rainfall, current moisture, irrigation
   availability, pumping capacity, canal allocation, or seasonal water sufficiency.
9. Missing SSURGO values produce `unknown` and lower evidence confidence; they never become
   automatic matches.
10. Do not turn this notebook's field comparisons into final rankings until the deterministic
    scoring and confidence policy is finalized and tested.

### Endpoint contract captured by this notebook

```text
POST https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest
Content-Type: application/json

{
  "query": "SELECT ...",
  "format": "JSON+COLUMNNAME"
}
```

SDA can be unavailable during its nightly update window, and the official service documents
a maximum of 100,000 rows for `post.rest`. Production code should use a timeout, bounded retry,
cache by coordinate/query version, and a clear unavailable/unknown result rather than silently
substituting a soil match.

Root-depth fallback behavior follows the [gSSURGO User Guide](https://www.nrcs.usda.gov/sites/default/files/2022-08/gSSURGO_UserGuide_July2020.pdf),
which uses 150 cm when no root-restricting zone is identified.